In [22]:
import re

def extract_qa_data(log_file_path):
    qa_list = []
    current_data = None
    
    with open(log_file_path, 'r') as file:
        content = file.read()
        
        # Split by the separator pattern
        sections = content.split("--------------------------------")
        
        for section in sections:
            # Look for question-solution blocks
            question_match = re.search(r'question: (.*?)(?=\nsolution:)', section, re.DOTALL)
            solution_match = re.search(r'solution: (.*?)(?=\nextracted answer:)', section, re.DOTALL)
            extracted_match = re.search(r'extracted answer: (.*?)(?=\nis correct:)', section, re.DOTALL)
            correct_match = re.search(r'is correct: (True|False)', section)
            accuracy_match = re.search(r'Accuracy: ([\d\.]+)', section)
            tokens_match = re.search(r'Number of generated tokens: (\d+)', section)
            error_match = re.search(r'Error steps count: (\d+)', section)
            
            # If we found a question, create a new entry
            if question_match:
                current_data = {
                    'question': question_match.group(1).strip(),
                    'solution': solution_match.group(1).strip() if solution_match else None,
                    'extracted_answer': extracted_match.group(1).strip() if extracted_match else None,
                    'is_correct': correct_match.group(1) == 'True' if correct_match else None,
                    'accuracy': float(accuracy_match.group(1)) if accuracy_match else None,
                    'token_count': int(tokens_match.group(1)) if tokens_match else None,
                    'error_steps': int(error_match.group(1)) if error_match else None
                }
                qa_list.append(current_data)
    
    # Also extract rewards and solutions lists
    rewards_lists = []
    for match in re.finditer(r'rewards: \[(.*?)\]', content):
        rewards_str = match.group(1)
        rewards = [float(x.strip()) for x in rewards_str.split(',')]
        rewards_lists.append(rewards)
    
    solutions_lists = []
    for match in re.finditer(r'found solutions: \[(.*?)\]', content):
        solutions_str = match.group(1)
        # Parse the solutions list, handling None values
        solutions = []
        for item in re.findall(r'(\'.*?\'|None)', solutions_str):
            if item == 'None':
                solutions.append(None)
            else:
                # Remove the quotes
                solutions.append(item.strip("'"))
        solutions_lists.append(solutions)
    
    return {
        'qa_data': qa_list,
        'rewards_lists': rewards_lists,
        'solutions_lists': solutions_lists
    }

def filter_high_reward_questions(processed_data, reward_threshold=0.9):
    """
    Filter the processed data to get only questions with rewards greater than the threshold.
    
    Args:
        processed_data: Dictionary containing 'qa_data', 'rewards_lists', and 'solutions_lists'
        reward_threshold: Minimum reward value (default 0.9)
        
    Returns:
        Dictionary with high-reward questions, solutions, and reward values
    """
    qa_data = processed_data['qa_data']
    rewards_lists = processed_data['rewards_lists']
    solutions_lists = processed_data['solutions_lists']
    
    high_reward_indices = []
    
    # Find indices of rewards lists where all values are above threshold
    for i, rewards in enumerate(rewards_lists):
        if all(reward > reward_threshold for reward in rewards):
            high_reward_indices.append(i)
    
    # Get the corresponding QA data, solutions, and rewards
    high_reward_qa = []
    high_reward_solutions = []
    high_reward_values = []
    
    for i in high_reward_indices:
        if i < len(qa_data):
            high_reward_qa.append(qa_data[i])
        if i < len(solutions_lists):
            high_reward_solutions.append(solutions_lists[i])
        if i < len(rewards_lists):
            high_reward_values.append(rewards_lists[i])
    
    return {
        'high_reward_qa': high_reward_qa,
        'high_reward_solutions': high_reward_solutions,
        'high_reward_values': high_reward_values
    }



In [31]:
# Example usage
log_file_path = '/satassdscratch/byuan48/inference-on-LLMs/output_ST.log'
data_ST = extract_qa_data(log_file_path)

print(f"Total questions processed: {len(data_ST['qa_data'])}")
print(f"Total rewards lists: {len(data_ST['rewards_lists'])}")
print(f"Total solutions lists: {len(data_ST['solutions_lists'])}")

# Print a sample question
if data['qa_data']:
    print("\nSample question data:")
    print(data['qa_data'][0])

# Print a sample rewards list
if data['rewards_lists']:
    print("\nSample rewards list:")
    print(data['rewards_lists'][0][:5])  # First 5 rewards

# Print a sample solutions list
if data['solutions_lists']:
    print("\nSample solutions list:")
    print(data['solutions_lists'][0][:5])  # First 5 solutions

for i in range(0, 100, 10):
    high_reward_data = filter_high_reward_questions(data, reward_threshold=i/100)
    print(f"Threshold: {i/100}, Number of questions: {len(high_reward_data['high_reward_qa'])}")

Total questions processed: 500
Total rewards lists: 500
Total solutions lists: 500

Sample question data:
{'question': 'Convert the point $(0,3)$ in rectangular coordinates to polar coordinates.  Enter your answer in the form $(r,\\theta),$ where $r > 0$ and $0 \\le \\theta < 2 \\pi.$', 'solution': '\\left(3, \\frac{\\pi}{2}\\right)', 'extracted_answer': '\\left( 3, \\frac{\\pi}{2} \\right)', 'is_correct': True, 'accuracy': None, 'token_count': None, 'error_steps': None}

Sample rewards list:
[0.921875, 0.93359375, 0.99609375, 0.01385498046875]

Sample solutions list:
['\\\\left(3, \\\\frac{\\\\pi}{2}\\\\right)', '\\\\left(3, \\\\frac{\\\\pi}{2}\\\\right)', '\\\\left(3, \\\\frac{\\\\pi}{2}\\\\right)', '(3,0)']
Threshold: 0.0, Number of questions: 368
Threshold: 0.1, Number of questions: 142
Threshold: 0.2, Number of questions: 134
Threshold: 0.3, Number of questions: 129
Threshold: 0.4, Number of questions: 122
Threshold: 0.5, Number of questions: 116
Threshold: 0.6, Number of question

In [29]:
# Example usage
log_file_path = '/satassdscratch/byuan48/inference-on-LLMs/output_WM.log'
data_WM = extract_qa_data(log_file_path)

print(f"Total questions processed: {len(data['qa_data'])}")
print(f"Total rewards lists: {len(data['rewards_lists'])}")
print(f"Total solutions lists: {len(data['solutions_lists'])}")

# Print a sample question
if data['qa_data']:
    print("\nSample question data:")
    print(data['qa_data'][0])

# Print a sample rewards list
if data['rewards_lists']:
    print("\nSample rewards list:")
    print(data['rewards_lists'][0][:5])  # First 5 rewards

# Print a sample solutions list
if data['solutions_lists']:
    print("\nSample solutions list:")
    print(data['solutions_lists'][0][:5])  # First 5 solutions

for i in range(0, 100, 10):
    high_reward_data = filter_high_reward_questions(data, reward_threshold=i/100)
    print(f"Threshold: {i/100}, Number of questions: {len(high_reward_data['high_reward_qa'])}")


Total questions processed: 500
Total rewards lists: 500
Total solutions lists: 500

Sample question data:
{'question': 'Convert the point $(0,3)$ in rectangular coordinates to polar coordinates.  Enter your answer in the form $(r,\\theta),$ where $r > 0$ and $0 \\le \\theta < 2 \\pi.$', 'solution': '\\left(3, \\frac{\\pi}{2}\\right)', 'extracted_answer': '\\left( 3, \\frac{\\pi}{2} \\right)', 'is_correct': True, 'accuracy': None, 'token_count': None, 'error_steps': None}

Sample rewards list:
[0.921875, 0.93359375, 0.99609375, 0.01385498046875]

Sample solutions list:
['\\\\left(3, \\\\frac{\\\\pi}{2}\\\\right)', '\\\\left(3, \\\\frac{\\\\pi}{2}\\\\right)', '\\\\left(3, \\\\frac{\\\\pi}{2}\\\\right)', '(3,0)']
Threshold: 0.0, Number of questions: 368
Threshold: 0.1, Number of questions: 142
Threshold: 0.2, Number of questions: 134
Threshold: 0.3, Number of questions: 129
Threshold: 0.4, Number of questions: 122
Threshold: 0.5, Number of questions: 116
Threshold: 0.6, Number of question

In [37]:
def find_different_correctness(data_ST, data_WM):
    """
    Compare two sets of QA data and find questions where the correctness differs.
    Also records reward information for each method.
    
    Args:
        data1: First dataset with 'qa_data' containing question information
        data2: Second dataset with 'qa_data' containing question information
        
    Returns:
        Dictionary with questions that have different correctness values and rewards in the two datasets
    """
    qa_data_ST = data_ST['qa_data']
    qa_data_WM = data_WM['qa_data']
    rewards_ST = data_ST['rewards_lists']
    rewards_WM = data_WM['rewards_lists']
    
    # Create dictionaries mapping questions to their data for faster lookup
    qa_dict_ST = {qa['question']: qa for qa in qa_data_ST}
    qa_dict_WM = {qa['question']: qa for qa in qa_data_WM}
    
    # Create dictionaries mapping questions to their rewards
    rewards_dict_ST = {qa['question']: rewards for qa, rewards in zip(qa_data_ST, rewards_ST)}
    rewards_dict_WM = {qa['question']: rewards for qa, rewards in zip(qa_data_WM, rewards_WM)}
    
    # Find questions that exist in both datasets
    common_questions = set(qa_dict_ST.keys()) & set(qa_dict_WM.keys())
    
    # Find questions with different correctness values
    different_correctness = []
    
    for question in common_questions:
        is_correct_ST = qa_dict_ST[question]['is_correct']
        is_correct_WM = qa_dict_WM[question]['is_correct']
        
        if is_correct_ST != is_correct_WM:
            different_correctness.append({
                'question': question,
                'data_ST': qa_dict_ST[question],
                'data_WM': qa_dict_WM[question],
                'rewards_ST': rewards_dict_ST[question],
                'rewards_WM': rewards_dict_WM[question]
            })
    
    return {
        'different_correctness': different_correctness,
        'count': len(different_correctness),
        'total_common_questions': len(common_questions)
    }

In [38]:
find_different_correctness(data_ST, data_WM)

{'different_correctness': [{'question': 'Express $555_{10}$ in base $5$.',
   'data_ST': {'question': 'Express $555_{10}$ in base $5$.',
    'solution': '42105',
    'extracted_answer': '4210_{5}',
    'is_correct': False,
    'accuracy': 0.6171875,
    'token_count': 525,
    'error_steps': 0},
   'data_WM': {'question': 'Express $555_{10}$ in base $5$.',
    'solution': '4210',
    'extracted_answer': '4210_{5}',
    'is_correct': True,
    'accuracy': None,
    'token_count': None,
    'error_steps': None},
   'rewards_ST': [0.30078125, 0.006683349609375, 0.0029296875, 0.15234375],
   'rewards_WM': [0.0235595703125, 0.58203125, 0.9921875, 0.26953125]},
  {'question': "Alice and Bob are playing a game. Alice starts first. On Alice's turn, she flips a coin. If she gets a heads, she wins. If not, it becomes Bob's turn. On Bob's turn, he flips a coin. If he gets a tails, he wins. If not, it becomes Alice's turn. What is the probability that Alice wins the game?",
   'data_ST': {'questio

In [39]:
def count_correct_answers(data):
    """
    Count the number of correct answers in a dataset.
    
    Args:
        data: Dataset with 'qa_data' containing question information
        
    Returns:
        Number of correct answers
    """
    correct_count = 0
    
    for qa in data['qa_data']:
        if qa['is_correct'] is True:
            correct_count += 1
    
    return correct_count

In [40]:
print(count_correct_answers(data_ST))
print(count_correct_answers(data_WM))

298
286


In [41]:
def compute_mean_reward(data):
    """
    Compute the mean of all rewards in a dataset.
    
    Args:
        data: Dataset with 'rewards_lists' containing lists of reward values
        
    Returns:
        Mean reward value
    """
    all_rewards = []
    
    # Flatten all reward lists into a single list
    for rewards_list in data['rewards_lists']:
        all_rewards.extend(rewards_list)
    
    # Compute the mean if there are rewards
    if all_rewards:
        mean_reward = sum(all_rewards) / len(all_rewards)
        return mean_reward
    else:
        return None
print(compute_mean_reward(data_ST))
print(compute_mean_reward(data_WM))




0.46692721462249753
0.4522572021484375
